# 00_03 Serverless Compute in Free Edition

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Understand where your code runs in Databricks Free Edition, what serverless means, and how it differs from classic compute.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Every performance, cost and configuration question in Databricks starts with the compute. Interviewers often ask about compute types, and you need to know what your Free Edition can and cannot show you. Many later lessons point out where serverless behaves differently (no <code>sparkContext</code>, no caching, restricted configs).
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Classic vs serverless compute</b><br>
<b>Compute</b> is the set of machines that run your code. Databricks has two broad styles. Free Edition gives you <b>serverless only</b>, with usage quotas. That is enough to learn Spark, Delta and Unity Catalog, but you cannot build custom clusters here.
</div>

| | Classic compute | Serverless compute |
|---|---|---|
| Who manages machines | You (choose size, workers, runtime) | Databricks |
| Startup time | Minutes | Seconds |
| Configuration | Many settings (node types, autoscaling, init scripts, Spark configs) | Minimal, many settings are fixed |
| Billing idea | Pay for VMs while running | Pay for usage |

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A team runs nightly ETL on job compute sized for the workload, while analysts use serverless SQL warehouses for quick queries. Choosing the right one is a cost and performance decision.
</div>

## Compute types (know these for interviews)

| Type | Used for |
|---|---|
| All-purpose compute | Interactive development, shared by users |
| Job compute | Automated jobs, created for a run and terminated after |
| SQL warehouse | SQL queries and dashboards |
| Serverless | Managed compute with no cluster configuration |
| Instance pools | Pre-warmed VMs to reduce classic cluster startup (paid workspaces) |

In Free Edition you will mainly use serverless notebooks and a serverless SQL warehouse.

## What serverless changes for your code

| On classic compute | On serverless |
|---|---|
| `spark.sparkContext` and RDDs available | Not available. Use the DataFrame and SQL APIs |
| `df.cache()` / `persist()` | Not supported |
| Most `spark.conf.set(...)` allowed | Only a short list of settings, such as `spark.sql.session.timeZone`, `spark.sql.shuffle.partitions`, `spark.sql.ansi.enabled` |
| Install libraries on the cluster | `%pip install` in the notebook, or the notebook's environment panel |
| DBFS root for files | Unity Catalog volumes (`/Volumes/...`) |

Serverless notebooks talk to Spark through **Spark Connect**, which is why driver-side objects such as `sparkContext` aren't exposed.

## 1. What am I running on?

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Prints the Spark version, Python version and the CPU count of the driver's Python process.<br><br>
<b>Why it matters</b><br>Gives a baseline for the environment. Numbers vary by environment, so note what you see.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Three lines with version and CPU information.
</div>

In [0]:
import os, platform

print("Spark version  : ", spark.version )
print("Python version : " , platform.python_version())
print("Driver CPU count (Python side) : ", os.cpu_count())

## 2. Reading a Spark setting safely

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Tries to read <code>spark.sql.shuffle.partitions</code> and handles the case where serverless doesn't allow it.<br><br>
<b>Why it matters</b><br>Serverless restricts many Spark configs. The try/except pattern lets you probe a setting without breaking the notebook.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A value (often <code>auto</code>) or a message that it can't be read.
</div>

In [0]:
try:
    print("shuffle partitions : ", spark.conf.get("spark.sql.shuffle.partitions"))
except Exception as e:
    print("Setting not readable here : ", e)

## 3. A small timing test

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Counts 10 million generated rows twice and measures how long each run takes.<br><br>
<b>Why it matters</b><br>Gives a rough feel for serverless speed. Running it twice shows the effect of warm-up.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Versions, the shuffle partitions setting, and two timings of a few seconds or less.
</div>

In [0]:
import os, platform, time

print("Spark version:", spark.version)
print("Python version:", platform.python_version())
try:
    print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
except Exception:
    print("shuffle partitions: not readable on serverless")

times = []
for _ in range(2):
    start = time.time()
    spark.range(10_000_000).count()
    times.append(round(time.time() - start, 2))
print("10M row count (run 1 / run 2):", times[0], "s /", times[1], "s")

## 4. Probe what serverless allows

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Tries three things that differ between classic and serverless compute: reading <code>sparkContext</code>, caching a DataFrame, and setting two configs.<br><br>
<b>Why it matters</b><br>Knowing these limits up front explains errors you'll meet in later lessons, and they make good interview talking points ("what's different about serverless?").
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
On serverless: <code>sparkContext</code> and <code>cache</code> report "not available", setting the session time zone works, and an unsupported config is rejected. On classic compute, everything succeeds. Write down what you see.
</div>

In [0]:
checks = {
    "sparkContext": lambda: spark.sparkContext.appName,
    "cache()": lambda: spark.range(10).cache().count(),
    "set session time zone": lambda: spark.conf.set("spark.sql.session.timeZone", "UTC"),
    "set autoBroadcastJoinThreshold": lambda: spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1"),
}
for name, check in checks.items():
    try:
        check()
        print(f"{name:<32} OK")
    except Exception as e:
        print(f"{name:<32} not available: {type(e).__name__}")

## My observations

```
Spark version: 4.2.0
Python version: 3.12.3
shuffle partitions: auto
10M row count (run 1 / run 2): 0.29 s / 0.37 s
```

- What I noticed: the first run includes warm-up, so the second run is usually faster. Serverless starts quickly and needs no cluster setup, but it exposes fewer settings than classic compute.
- Probe results (section 4): (add yours here)

## Free Edition limits to remember (verified in the UI)

- Serverless compute only
- Usage quotas apply
- No custom clusters, init scripts or instance pools
- Some enterprise features are unavailable

Date I verified this: 2026-10-05

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>[NOT_SUPPORTED_WITH_SERVERLESS]</code> / <code>sparkContext</code> errors</b><br>
Serverless uses Spark Connect. Use the DataFrame, SQL and <code>spark.conf</code> APIs instead of <code>sparkContext</code> and RDDs.<br><br>
**⛔ Problem: <code>CONFIG_NOT_AVAILABLE</code> when setting a config**<br>Only some Spark configs can be set on serverless. Remove the setting, or use a supported alternative such as a query hint.<br><br>
**⛔ Problem: <code>df.cache()</code> fails**<br>Caching isn't supported on serverless. Write intermediate results to a table if you need to reuse them.<br><br>
<b>⛔ Problem: notebook stops or slows down after heavy use</b><br>Free Edition has usage quotas. Keep sample data small and stop long-running work.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is serverless compute?</b><br>
Compute where Databricks manages the infrastructure: no cluster sizing, fast startup, and billing based on usage.<br><br>

<b>🎤 2. All-purpose vs job compute?</b><br>
All-purpose is for interactive work and can be shared. Job compute is created for an automated run and terminates afterward, which is cheaper for production.<br><br>

<b>🎤 3. Why use job compute for production instead of an all-purpose cluster?</b><br>
Lower cost, isolation from interactive workloads, and a clean environment for each run.<br><br>

<b>🎤 4. What are instance pools?</b><br>
Pools of idle, pre-warmed VMs that reduce classic cluster startup time.<br><br>

<b>🎤 5. What is the trade-off of serverless?</b><br>
Less control over configuration in exchange for simplicity and fast startup.<br><br>

<b>🎤 6. Why can't you use <code>sparkContext</code> or RDDs on serverless notebooks?</b><br>
Serverless notebooks use Spark Connect, a client-server protocol where the notebook sends DataFrame and SQL plans to a remote Spark. Driver-side objects such as <code>sparkContext</code> aren't exposed to the client.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A scheduled nightly pipeline currently runs on a shared all-purpose cluster. What is the most cost-effective compute choice for it?</b><br>
A. Keep the all-purpose cluster running 24/7<br>
B. Job compute (or serverless jobs compute), created for each run<br>
C. A SQL warehouse<br>
D. A larger all-purpose cluster<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Job compute starts for the run and terminates afterwards, so you don't pay for idle time, and the run is isolated from interactive users.</details><br><br>

<b>Q2. Which is a benefit of serverless compute compared with classic compute?</b><br>
A. Full control over node types and init scripts<br>
B. Fast startup with infrastructure managed by Databricks<br>
C. Access to <code>sparkContext</code> and RDD APIs<br>
D. Ability to install JVM libraries on the driver<br>
<details><summary><b>Show answer</b></summary><b>B.</b> The other options describe classic compute.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Time <code>spark.range(50_000_000).count()</code> three times and record the results</li><li>Read <code>spark.sql.session.timeZone</code>, set it to <code>Asia/Kolkata</code>, read it again, then set it back to the original</li><li>In one sentence: when would you choose a SQL warehouse instead of a notebook on serverless compute?</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
import time

# 1. Three timings
for i in range(3):
    start = time.time()
    spark.range(50_000_000).count()
    print(f"run {i + 1}: {time.time() - start:.2f} s")

# 2. Read, change, read, restore
original = spark.conf.get("spark.sql.session.timeZone")
spark.conf.set("spark.sql.session.timeZone", "Asia/Kolkata")
print("changed to:", spark.conf.get("spark.sql.session.timeZone"))
spark.conf.set("spark.sql.session.timeZone", original)
print("restored to:", spark.conf.get("spark.sql.session.timeZone"))

# 3. A SQL warehouse is the better choice for SQL-only analytics, dashboards and BI tools,
#    where many users run concurrent queries.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Compute types: all-purpose, job, SQL warehouse, serverless, instance pools</li><li>Free Edition = serverless only, with quotas</li><li>Serverless notebooks use Spark Connect: no <code>sparkContext</code>, no RDDs, no <code>cache()</code>, limited configs</li><li>Use job compute (or serverless jobs) for production runs, not all-purpose clusters</li><li>Probe settings with try/except and record what your environment allows</li></ul>
</div>

| Task | Code |
|---|---|
| Spark version | `spark.version` |
| Read a config | `spark.conf.get("spark.sql.shuffle.partitions")` |
| Set a config | `spark.conf.set("spark.sql.session.timeZone", "UTC")` |
| Time a query | `start = time.time(); df.count(); time.time() - start` |
| Install a library | `%pip install <package>` |

## Git commit

```
git add 00_platform_tour/00_03_serverless_compute_free_edition.ipynb
git commit -m "restyle 00_03 serverless compute to lesson standard"
```